# WAV_L1 paired confirmation — seed 123
Run this notebook on one Colab GPU account. It trains **only seed 123** from the existing seed-matched D0 checkpoint. Seed 42 is frozen repository evidence. Locked test remains closed. Seed 2026 should run in the separate notebook, optionally in parallel on another account.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, json, os, shutil, subprocess, sys, tarfile, time, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
SEED=123; BRANCH='agent/wav1-mechanism-factorization'; REPO=Path('/content/coffee-bean-detection')
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(1,4):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    shutil.rmtree(REPO,ignore_errors=True)
    if attempt==3: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
REQ=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-acmc-paired-confirmation-v1/D0_base/D0_seed123/weights/best.pt',
)
PROJECT=resolve_drive_project_root(required_relative_paths=REQ)
ARCHIVE=require_project_artifact(PROJECT,REQ[0]); D0=require_project_artifact(PROJECT,REQ[1])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'faruq_grouped_summary.json').is_file()
assert not (DATA/'test').exists(), 'STOP: locked test tidak boleh tersedia'
OUT=PROJECT/'experiments/faruq-v3-wav-l1-paired-confirmation-v1'/'seed123'
OUT.mkdir(parents=True,exist_ok=True)
print('PROJECT:',PROJECT); print('D0:',D0); print('OUTPUT:',OUT)
print('FROZEN PROTOCOL:',REPO/'docs/FARUQ_V3_WAV_L1_PAIRED_CONFIRMATION_PROTOCOL_2026-08-19.md')


In [ ]:
CMD=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_wav_l1_confirmation_seed',
 '--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),
 '--d0-checkpoint',str(D0),'--output-root',str(OUT),'--seed',str(SEED),'--device','0','--authorize-training']
LOG=OUT/f'WAV_L1_seed{SEED}_confirmation.log'; RESULT=OUT/'val_reports'/f'WAV_L1_seed{SEED}_result.json'
print('MENJALANKAN:', ' '.join(CMD),flush=True)
with LOG.open('a',encoding='utf-8',buffering=1) as stream:
    process=subprocess.Popen(CMD,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    seen=-1
    while process.poll() is None:
        csv=OUT/'WAV_L1'/f'WAV_L1_seed{SEED}'/'results.csv'
        n=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if n!=seen: print(f'WAV_L1 seed {SEED}: {n}/50 epoch | log={LOG}',flush=True); seen=n
        time.sleep(120)
    code=process.wait()
if code:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-160:])); raise RuntimeError(f'seed {SEED} gagal: {code}')
assert RESULT.is_file(),RESULT
result=json.loads(RESULT.read_text(encoding='utf-8')); assert result['seed']==SEED and result['evaluation_split']=='val' and result['test_images_accessed'] is False
print(json.dumps(result,indent=2))
print('SEED 123 SELESAI. Jangan buka locked test. Setelah seed2026 selesai, jalankan notebook Pair Decision.')
